In [1]:
import pandas as pd

In [2]:
tadf_pl_ds = pd.read_excel("tadf_train_corrected_for_annotation.xlsx", sheet_name="4_ml_cols_only")
tadf_pl_ds.head()

,_id,doi,solvent,standard_value,smiles,corrected
0,0,https://doi.org/10.1016/j.cej.2022.135775,toluene,638.0,N1=CC=C(C=C1)C=1C=C2N=C3C(=NC2=CC1C1=CC=NC=C1)...,1
1,1,https://doi.org/10.1016/j.cej.2022.135775,toluene,639.0,N1=CC=C(C=C1)C=1C=C2N=C3C4=C(C5=C(C3=NC2=CC1C1...,1
2,2,https://doi.org/10.1016/j.dyepig.2023.111250,toluene,465.0,C1(=CC=CC=C1)C1=NC(=NC(=N1)C1=CC=CC=C1)C1=C(C=...,1
3,17,https://doi.org/10.1016/j.cej.2020.127591,toluene,415.0,C1=CC=CC=2C=3C=CC4=C(C3N(C12)C1=NC=C(N=C1)N1C=...,1
4,18,https://doi.org/10.1016/j.cej.2020.127591,toluene,444.0,C1=C2C=3C4=C(C=CC3N(C2=CC=C1)C1=NC=C(N=C1)N1C2...,1


In [2]:
from rdkit import Chem

In [4]:
# add canonical smiles
tadf_pl_ds["canonical_smiles"] = tadf_pl_ds["smiles"].apply(Chem.CanonSmiles)

In [3]:
solvent2smiles={
    'toluene': 'CC1=CC=CC=C1',
    'thf': 'O1CCCC1',
    '2-methf': 'CC1OCCC1',
    'hexane': 'CCCCCC',
    'dichloromethane': 'ClCCl',
    'ch2cl2': 'ClCCl',
    'chcl3': 'ClC(Cl)Cl',
    'tetrahydrofuran': 'O1CCCC1',
    'chloroform': 'C(Cl)(Cl)Cl',
    'cyclohexane': 'C1CCCCC1',
    'dcm': 'ClCCl',
    'ch3cn': 'C(C)#N',
    'methylcyclohexane': 'CC1CCCCC1',
    '2-me-thf': 'CC1OCCC1',
    'dmso': 'CS(=O)C',
    'meoh': 'CO',
    'dmf': 'CN(C=O)C',
    'phme': 'CC1=CC=CC=C1',
    'mecn': 'C(C)#N',
    'acetonitrile': 'C(C)#N',
}

In [4]:
for k, v in solvent2smiles.items():
    solvent2smiles[k] = Chem.CanonSmiles(v)

In [5]:
def get_solvent_smiles(solvent):
    if not solvent or type(solvent) != str:
        return None
    return solvent2smiles[solvent.lower()]
# tadf_pl_ds["solvent_smiles"] = tadf_pl_ds["solvent"].apply(get_solvent_smiles)

In [8]:
tadf_pl_by_canonsmiles = tadf_pl_ds.groupby(["canonical_smiles", "solvent_smiles"])

In [23]:
group_sizes = tadf_pl_by_canonsmiles.size().sort_values(ascending=False)
group_sizes_large = group_sizes[group_sizes > 1]

In [26]:
Chem.CanonSmiles("O=C(c1c2cc(c3ccc(N(c4ccc(OCCCCCCN5c6ccccc6c7ccccc75)cc4)c8ccc(OCCCCCCN9c%10ccccc%10c%11ccccc%119)cc8)cc3)cc1)c%12cc(c%13ccc(N(c%14ccc(OCCCCCCN%15c%16ccccc%16c%17ccccc%17%15)cc%14)c%18ccc(OCCCCCCN%19c%20ccccc%20c%21ccccc%21%19)cc%18)cc%13)ccc%12C2=O")

'O=C1c2ccc(-c3ccc(N(c4ccc(OCCCCCCn5c6ccccc6c6ccccc65)cc4)c4ccc(OCCCCCCn5c6ccccc6c6ccccc65)cc4)cc3)cc2C(=O)c2ccc(-c3ccc(N(c4ccc(OCCCCCCn5c6ccccc6c6ccccc65)cc4)c4ccc(OCCCCCCn5c6ccccc6c6ccccc65)cc4)cc3)cc21'

In [24]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    for group_key in group_sizes_large.index:
        print(f"Size: {group_sizes[group_key]}")
        display(tadf_pl_by_canonsmiles.get_group(group_key).drop(columns=["smiles", "solvent_smiles"]))

Size: 4


,_id,doi,solvent,standard_value,corrected,canonical_smiles
442,377,https://doi.org/10.1039/D1TC05696D,Toluene,476.0,0,N#Cc1cc(-c2cccc3c2[nH]c2ccccc23)c(-c2cccc3c2[n...
443,378,https://doi.org/10.1039/D1TC05696D,Toluene,496.0,0,N#Cc1cc(-c2cccc3c2[nH]c2ccccc23)c(-c2cccc3c2[n...
444,380,https://doi.org/10.1039/D1TC05696D,Toluene,484.0,0,N#Cc1cc(-c2cccc3c2[nH]c2ccccc23)c(-c2cccc3c2[n...
445,381,https://doi.org/10.1039/D1TC05696D,Toluene,477.0,0,N#Cc1cc(-c2cccc3c2[nH]c2ccccc23)c(-c2cccc3c2[n...


Size: 4


,_id,doi,solvent,standard_value,corrected,canonical_smiles
330,177,https://doi.org/10.1016/j.dyepig.2021.109580,toluene,578.0,0,CC1(C)c2ccccc2N(c2ccc3nc4c5ccccc5c5ccccc5c4nc3...
420,334,https://doi.org/10.1039/D0TC01995J,toluene,567.0,0,CC1(C)c2ccccc2N(c2ccc3nc4c5ccccc5c5ccccc5c4nc3...
525,493,https://doi.org/10.1039/D3TC02352D,PhMe,595.0,0,CC1(C)c2ccccc2N(c2ccc3nc4c5ccccc5c5ccccc5c4nc3...
526,494,https://doi.org/10.1039/D3TC02352D,PhMe,568.0,0,CC1(C)c2ccccc2N(c2ccc3nc4c5ccccc5c5ccccc5c4nc3...


Size: 4


...

,_id,doi,solvent,standard_value,corrected,canonical_smiles
323,164,https://doi.org/10.1016/j.orgel.2018.11.018,CH2Cl2,512.0,0,CC(C)(C)c1ccc2c(c1)c1cc(C(C)(C)C)ccc1n2-c1ccc2...
324,165,https://doi.org/10.1016/j.orgel.2018.11.018,CH2Cl2,502.0,0,CC(C)(C)c1ccc2c(c1)c1cc(C(C)(C)C)ccc1n2-c1ccc2...


Size: 2


,_id,doi,solvent,standard_value,corrected,canonical_smiles
351,219,https://doi.org/10.1016/j.orgel.2019.03.044,DMF,423.0,0,FC(F)(F)c1ccc(-n2c(-c3ccc(-c4ccc(-n5c6ccccc6c6...
352,220,https://doi.org/10.1016/j.orgel.2019.03.044,DMF,440.0,0,FC(F)(F)c1ccc(-n2c(-c3ccc(-c4ccc(-n5c6ccccc6c6...


Size: 2


,_id,doi,solvent,standard_value,corrected,canonical_smiles
282,93,https://doi.org/10.1016/j.dyepig.2022.110634,toluene,516.0,0,c1cnc2c(c1)c1nc3cc(-n4c5ccccc5c5ccccc54)c(-n4c...
283,94,https://doi.org/10.1016/j.dyepig.2022.110634,toluene,555.0,0,c1cnc2c(c1)c1nc3cc(-n4c5ccccc5c5ccccc54)c(-n4c...


In [13]:
df = pd.read_excel("tadf_train_corrected_for_annotation.xlsx", sheet_name="6_resolved_duplicates")

In [14]:
def canonical_smiles(smiles):
    try:
        canon_smiles = Chem.CanonSmiles(smiles)
    except Exception as e:
        print(f"Error processing SMILES: {smiles}, Error: {e}")
        return None
    return canon_smiles
    

In [15]:
df["_temp_canonical_smiles"] = df["canonical_smiles"].apply(canonical_smiles)

In [16]:
df = df.drop(columns=["canonical_smiles"])
df = df.rename(columns={"_temp_canonical_smiles": "canonical_smiles"})

In [17]:
df["solvent_smiles"] = df["solvent"].apply(get_solvent_smiles)

In [18]:
with pd.ExcelWriter("tadf_train_corrected_for_annotation.xlsx", engine="openpyxl", mode="a") as writer:
    df.to_excel(writer, sheet_name="7_resolved_duplicates", index=False)